# Part 1: Data Cleaning & Validation


In [1]:
from contextlib import closing
from datetime import datetime, timedelta
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import papermill as pm
from IPython.display import display
from tqdm import tqdm

### 1.1 Load and Inspect the Data

- Load the `ohlc_hw` table from `data/data.db`
- Display basic statistics (`.info()`, `.describe()`)
- Identify the columns and their meanings
- Show the date range and list of tickers

In [2]:
# Use the local database copy stored in this homework folder.
database = Path('data/data.db')

if not database.is_file():
    raise FileNotFoundError(f'Download data.db from bCourses and place it at {Path.cwd() / database}')

print(f'Local homework database: {database.resolve()}')

Local homework database: /Users/philipsyan/Desktop/Pre-MFE Python/github_work/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/Xiaofei_Yan/data/data.db


In [3]:
# Open the database in read-only mode so this inspection cannot alter the source tables.
database_uri = database.resolve().as_uri() + '?mode=ro'
with closing(sqlite3.connect(database_uri, uri=True)) as connection:
    ohlc_hw = pd.read_sql_query(
        'SELECT * FROM ohlc_hw',
        connection,
        parse_dates=['ts'],
    )

print(f'Shape: {ohlc_hw.shape[0]:,} rows × {ohlc_hw.shape[1]} columns')
display(ohlc_hw.head())

Shape: 397,711 rows × 8 columns


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [4]:
# .info() reports column types and non-null counts.
ohlc_hw.info()

# .describe() summarizes the numeric columns.
display(ohlc_hw.describe())

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[us]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(1)
memory usage: 24.3 MB


,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292486,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08


### Column meanings

| Column | Meaning |
| --- | --- |
| `ts` | Observation date. |
| `ticker` | Stock symbol identifying the security. |
| `open` | Raw opening price for the date. |
| `high` | Raw highest price for the date. |
| `low` | Raw lowest price for the date. |
| `close` | Raw closing price for the date. |
| `adj_close` | Vendor-provided adjusted closing price. |
| `volume` | Reported trading volume for the date. |

The table's physical column order may differ from this documentation; the names and meanings are what matter.

In [5]:
date_range = (ohlc_hw['ts'].min(), ohlc_hw['ts'].max())

# Follow Lecture 4/1's ticker-text cleanup: remove HTML, trim, and uppercase.
# Clean a copy for display; keep the original ticker column unchanged.
tickers = (ohlc_hw['ticker'].astype('string')
           .str.replace(r'<[^>]+>', '', regex=True)
           .str.strip().str.upper())
tickers = sorted(tickers.dropna().unique().tolist())

print(f'Date range: {date_range[0].date()} to {date_range[1].date()}')
print(f'Number of distinct tickers: {len(tickers)}')
print('Tickers:')
print(tickers)


Date range: 1962-01-02 to 2024-12-06
Number of distinct tickers: 30
Tickers:
['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


### 1.2 Detect Data Quality Issues

Write functions to detect:

#### Missing Data

- Identify missing values in each column
- Detect missing trading days (gaps in the time series)
- Create a summary: "Ticker X is missing Y trading days between [start] and [end]"

For this assignment, the supplied `ohlc` table's dates for each ticker are an acceptable reference calendar over that ticker's supplied history. Alternatively, document an exchange-calendar and active-history assumption. Business weekdays alone include exchange holidays. Report zero gaps when appropriate, and demonstrate your detector on a tiny example with one deliberately removed date. Detecting absent rows does not require inventing prices for them.

The homework table's ticker labels contain HTML markup. The detector normalizes labels for matching while leaving the original `ohlc_hw` data unchanged.


In [6]:
def missing_value_summary(frame):
    """Count missing cells in each column."""
    return frame.isna().sum()

display(missing_value_summary(ohlc_hw))

# Reference keys provide expected observations for each ticker.
with closing(sqlite3.connect(database.resolve().as_uri() + '?mode=ro', uri=True)) as connection:
    reference_keys = pd.read_sql_query(
        'SELECT ticker, ts FROM ohlc ORDER BY ticker, ts',
        connection,
        parse_dates=['ts'],
    )

display(reference_keys.head())


ts               0
open             0
high             0
low              0
close            0
adj_close    19855
volume           0
ticker           0
dtype: int64

,ticker,ts
0,AAPL,1980-12-12
1,AAPL,1980-12-15
2,AAPL,1980-12-16
3,AAPL,1980-12-17
4,AAPL,1980-12-18


The counts above describe missing cells in existing rows. Missing trading dates are checked separately against the supplied reference calendar below.


In [7]:
def missing_sessions(frame, expected_dates):
    """Return expected dates absent from one ticker's observations."""
    observed = pd.DatetimeIndex(frame['ts'].dropna().unique())
    return pd.DatetimeIndex(expected_dates).difference(observed).sort_values()


In [8]:
# Normalize copies of the ticker labels for matching; keep ohlc_hw unchanged.
homework_calendar = ohlc_hw[['ticker', 'ts']].copy()
homework_calendar['ticker'] = (
    homework_calendar['ticker'].astype('string')
    .str.replace(r'<[^>]*>', '', regex=True).str.strip().str.upper()
)
reference_calendar = reference_keys.copy()
reference_calendar['ticker'] = (
    reference_calendar['ticker'].astype('string')
    .str.replace(r'<[^>]*>', '', regex=True).str.strip().str.upper()
)

summary_rows = []
missing_day_rows = []

for ticker, ticker_data in homework_calendar.groupby('ticker', sort=True):
    start_date = ticker_data['ts'].min()
    end_date = ticker_data['ts'].max()
    expected_dates = reference_calendar.loc[
        reference_calendar['ticker'].eq(ticker)
        & reference_calendar['ts'].between(start_date, end_date),
        'ts',
    ]

    if expected_dates.empty:
        summary = f'Ticker {ticker} has no dates in the supplied ohlc reference calendar.'
        missing_count = pd.NA
    else:
        gap_dates = missing_sessions(ticker_data, expected_dates)
        missing_count = len(gap_dates)
        summary = (
            f'Ticker {ticker} is missing {missing_count} trading days between '
            f'{start_date:%Y-%m-%d} and {end_date:%Y-%m-%d}.'
        )
        for gap_date in gap_dates:
            missing_day_rows.append({
                'ticker': ticker,
                'issue_type': 'missing_trading_day',
                'issue_date': gap_date,
                'description': 'Date is in the ohlc reference calendar but absent from ohlc_hw.',
                'severity': 'medium',
            })

    summary_rows.append({
        'ticker': ticker,
        'missing_sessions': missing_count,
        'summary': summary,
    })

missing_day_summary = pd.DataFrame(summary_rows)
issue_columns = ['ticker', 'issue_type', 'issue_date', 'description', 'severity']
missing_day_issues = pd.DataFrame(missing_day_rows, columns=issue_columns)
display(missing_day_summary)


,ticker,missing_sessions,summary
0,AAPL,0,Ticker AAPL is missing 0 trading days between ...
1,AMGN,0,Ticker AMGN is missing 0 trading days between ...
2,AMZN,0,Ticker AMZN is missing 0 trading days between ...
3,AXP,0,Ticker AXP is missing 0 trading days between 1...
4,BA,0,Ticker BA is missing 0 trading days between 19...
5,CAT,0,Ticker CAT is missing 0 trading days between 1...
6,CRM,0,Ticker CRM is missing 0 trading days between 2...
7,CSCO,0,Ticker CSCO is missing 0 trading days between ...
8,CVX,0,Ticker CVX is missing 0 trading days between 1...
9,DIS,0,Ticker DIS is missing 0 trading days between 1...


A small injected example checks that the detector finds an absent date without assuming every weekday is a trading day. The expected dates are supplied explicitly, just as the `ohlc` table supplies the calendar for the real data.


In [9]:
demo_reference_calendar = pd.DataFrame({
    'ticker': ['DEMO_A'] * 4,
    'ts': pd.to_datetime(['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05']),
})
demo_observations = demo_reference_calendar.drop(index=1).reset_index(drop=True)
demo_gap_dates = missing_sessions(demo_observations, demo_reference_calendar['ts'])

print(
    f"DEMO_A is missing {len(demo_gap_dates)} trading day between "
    f"{demo_reference_calendar['ts'].min():%Y-%m-%d} and "
    f"{demo_reference_calendar['ts'].max():%Y-%m-%d}: {demo_gap_dates.tolist()}"
)
assert demo_gap_dates.tolist() == [pd.Timestamp('2024-01-03')]


DEMO_A is missing 1 trading day between 2024-01-02 and 2024-01-05: [Timestamp('2024-01-03 00:00:00')]


In [10]:
issue_columns = ['ticker', 'issue_type', 'issue_date', 'description', 'severity']
missing_value_rows = []

for column in ohlc_hw.columns:
    missing_rows = ohlc_hw.loc[ohlc_hw[column].isna(), ['ticker', 'ts']].copy()
    if missing_rows.empty:
        continue

    missing_rows['ticker'] = (
        missing_rows['ticker'].astype('string')
        .str.replace(r'<[^>]*>', '', regex=True).str.strip().str.upper()
        .fillna('UNKNOWN')
    )
    missing_rows['issue_type'] = 'missing_value'
    missing_rows['issue_date'] = pd.to_datetime(missing_rows.pop('ts'), errors='coerce')
    missing_rows['description'] = f'Missing value in {column}.'
    missing_rows['severity'] = 'medium' if column == 'volume' else 'high'
    missing_value_rows.append(missing_rows[issue_columns])

if missing_value_rows:
    missing_value_issues = pd.concat(missing_value_rows, ignore_index=True)
else:
    missing_value_issues = pd.DataFrame(columns=issue_columns)

missing_data_issues = pd.concat(
    [missing_value_issues, missing_day_issues],
    ignore_index=True,
)
print(f'Missing-data issue rows: {len(missing_data_issues):,}')
display(
    missing_data_issues
    .groupby(['issue_type', 'description', 'severity'], dropna=False)
    .size()
    .rename('issue_count')
    .reset_index()
)
display(missing_data_issues.head(10))


Missing-data issue rows: 19,855


,issue_type,description,severity,issue_count
0,missing_value,Missing value in adj_close.,high,19855


,ticker,issue_type,issue_date,description,severity
0,KO,missing_value,2024-04-16 00:00:00,Missing value in adj_close.,high
1,NVDA,missing_value,2018-06-12 00:00:00,Missing value in adj_close.,high
2,HD,missing_value,2013-04-16 00:00:00,Missing value in adj_close.,high
3,PG,missing_value,2015-03-20 00:00:00,Missing value in adj_close.,high
4,JPM,missing_value,2018-03-13 00:00:00,Missing value in adj_close.,high
5,NVDA,missing_value,2000-05-15 00:00:00,Missing value in adj_close.,high
6,CSCO,missing_value,2001-10-02 00:00:00,Missing value in adj_close.,high
7,MMM,missing_value,1975-07-03 00:00:00,Missing value in adj_close.,high
8,IBM,missing_value,2019-06-21 00:00:00,Missing value in adj_close.,high
9,CSCO,missing_value,2006-01-09 00:00:00,Missing value in adj_close.,high


#### Outliers

- Calculate daily returns: `(close - prev_close) / prev_close`
- Flag `abs(return - mean_return) > 3 * std_return` within each ticker, using the sample standard deviation (`ddof=1`); flag both tails. Exclude undefined returns and treat zero/undefined standard deviation explicitly. This retrospective screen is a review flag, not a deletion rule.
- Flag any day where: `high < close` or `low > close` (impossible!)
- Flag nonpositive or non-finite equity prices, including adjusted close; raw OHLC bounds do not apply to adjusted close


In [11]:
# Normalize ticker labels before duplicate checks and grouping, as required by the assignment.
# Keep ohlc_hw unchanged; this normalized copy is used for diagnosis.
normalized_source = ohlc_hw.copy()
normalized_source['ticker'] = (normalized_source['ticker'].astype('string')
                               .str.replace(r'<[^>]+>', '', regex=True)
                               .str.strip().str.upper())

# Record extra exact copies, then remove them before calculating returns.
exact_duplicate_mask = normalized_source.duplicated(keep='first')
duplicate_issue_rows = normalized_source.loc[exact_duplicate_mask, ['ticker', 'ts']].copy()
duplicate_issue_rows = duplicate_issue_rows.rename(columns={'ts': 'issue_date'})
duplicate_issue_rows['issue_type'] = 'exact_duplicate'
duplicate_issue_rows['description'] = 'Exact duplicate record removed before calculations.'
duplicate_issue_rows = duplicate_issue_rows[['ticker', 'issue_type', 'issue_date', 'description']]

outlier_data = normalized_source.drop_duplicates().sort_values(['ticker', 'ts']).reset_index(drop=True)
valid_close = outlier_data['close'].where(
    np.isfinite(outlier_data['close']) & outlier_data['close'].gt(0)
)
outlier_data['daily_return'] = valid_close.groupby(outlier_data['ticker']).pct_change(fill_method=None)

returns_by_ticker = outlier_data['daily_return'].groupby(outlier_data['ticker'])
mean_return = returns_by_ticker.transform('mean')
std_return = returns_by_ticker.transform('std', ddof=1)

price_columns = ['open', 'high', 'low', 'close', 'adj_close']
invalid_price = (
    (~np.isfinite(outlier_data[price_columns])).any(axis=1)
    | outlier_data[price_columns].le(0).any(axis=1)
)

outlier_masks = {
    'extreme_return': (
        outlier_data['daily_return'].notna()
        & std_return.notna()
        & std_return.gt(0)
        & (outlier_data['daily_return'] - mean_return).abs().gt(3 * std_return)
    ),
    'high_below_close': outlier_data['high'].lt(outlier_data['close']),
    'low_above_close': outlier_data['low'].gt(outlier_data['close']),
    'invalid_price': invalid_price,
}

outlier_descriptions = {
    'extreme_return': 'Daily return is more than 3 sample standard deviations from the ticker mean; review only.',
    'high_below_close': 'High is below close.',
    'low_above_close': 'Low is above close.',
    'invalid_price': 'A raw or adjusted equity price is non-finite or nonpositive.',
}
outlier_issue_rows = []
for issue_type, issue_mask in outlier_masks.items():
    issue_rows = outlier_data.loc[issue_mask, ['ticker', 'ts']].rename(columns={'ts': 'issue_date'}).copy()
    issue_rows['issue_type'] = issue_type
    issue_rows['description'] = outlier_descriptions[issue_type]
    outlier_issue_rows.append(issue_rows[['ticker', 'issue_type', 'issue_date', 'description']])

outlier_issues = pd.concat(outlier_issue_rows, ignore_index=True)
display(outlier_issues.groupby('issue_type').size().rename('issue_count').reset_index())
display(outlier_issues.head(10))


,issue_type,issue_count
0,extreme_return,5056
1,invalid_price,18379


,ticker,issue_type,issue_date,description
0,AAPL,extreme_return,1980-12-26,Daily return is more than 3 sample standard de...
1,AAPL,extreme_return,1981-04-02,Daily return is more than 3 sample standard de...
2,AAPL,extreme_return,1981-09-25,Daily return is more than 3 sample standard de...
3,AAPL,extreme_return,1982-03-19,Daily return is more than 3 sample standard de...
4,AAPL,extreme_return,1982-11-22,Daily return is more than 3 sample standard de...
5,AAPL,extreme_return,1982-11-30,Daily return is more than 3 sample standard de...
6,AAPL,extreme_return,1983-01-20,Daily return is more than 3 sample standard de...
7,AAPL,extreme_return,1983-04-20,Daily return is more than 3 sample standard de...
8,AAPL,extreme_return,1983-07-26,Daily return is more than 3 sample standard de...
9,AAPL,extreme_return,1983-08-31,Daily return is more than 3 sample standard de...


#### Inconsistencies

- Verify that `low <= open <= high` and `low <= close <= high`
- Check for zero volume days (could indicate stale data)


For dates with a valid matched `ohlc` reference, this notebook also checks that `adj_close / close` agrees with the date-specific reference adjustment factor. This additional check was added after the data showed positive but corrupted adjusted-close values; it does not infer or forward-fill factors for unmatched dates.


In [12]:
# Follow Lecture 4/1's OHLC-bounds and zero-volume checks.
open_outside_range = outlier_data['open'].lt(outlier_data['low']) | outlier_data['open'].gt(outlier_data['high'])
close_outside_range = outlier_data['close'].lt(outlier_data['low']) | outlier_data['close'].gt(outlier_data['high'])
zero_volume = outlier_data['volume'].eq(0)

# Check existing adjusted-close factors against matched dates in the supplied ohlc reference.
with closing(sqlite3.connect(database.resolve().as_uri() + '?mode=ro', uri=True)) as connection:
    reference_prices = pd.read_sql_query(
        'SELECT ticker, ts, close, adj_close FROM ohlc', connection, parse_dates=['ts']
    )
reference_prices['ticker'] = (reference_prices['ticker'].astype('string')
                               .str.replace(r'<[^>]+>', '', regex=True)
                               .str.strip().str.upper())
valid_reference_factor = (
    np.isfinite(reference_prices['close']) & reference_prices['close'].gt(0)
    & np.isfinite(reference_prices['adj_close']) & reference_prices['adj_close'].gt(0)
)
reference_prices['adjustment_factor'] = np.nan
reference_prices.loc[valid_reference_factor, 'adjustment_factor'] = (
    reference_prices.loc[valid_reference_factor, 'adj_close']
    / reference_prices.loc[valid_reference_factor, 'close']
)
reference_factors_for_check = reference_prices[['ticker', 'ts', 'adjustment_factor']]
factor_check = outlier_data.merge(
    reference_factors_for_check, on=['ticker', 'ts'], how='left', validate='many_to_one'
)
current_factor = factor_check['adj_close'] / factor_check['close']
can_compare_factor = (
    np.isfinite(factor_check['close']) & factor_check['close'].gt(0)
    & np.isfinite(factor_check['adj_close']) & factor_check['adj_close'].gt(0)
    & np.isfinite(factor_check['adjustment_factor']) & factor_check['adjustment_factor'].gt(0)
)
adj_close_factor_mismatch = can_compare_factor & ~np.isclose(
    current_factor, factor_check['adjustment_factor'], rtol=1e-6, atol=1e-8
)

inconsistency_masks = {
    'open_outside_ohlc_range': open_outside_range,
    'close_outside_ohlc_range': close_outside_range,
    'zero_volume': zero_volume,
    'adj_close_factor_mismatch': adj_close_factor_mismatch,
}
inconsistency_descriptions = {
    'open_outside_ohlc_range': 'Open is outside the low-to-high range.',
    'close_outside_ohlc_range': 'Close is outside the low-to-high range.',
    'zero_volume': 'Volume is zero; check whether this is a stale-data issue.',
    'adj_close_factor_mismatch': 'Adjusted-close factor differs from the matched date-specific ohlc reference factor.',
}
inconsistency_issue_rows = []
for issue_type, issue_mask in inconsistency_masks.items():
    issue_rows = outlier_data.loc[issue_mask, ['ticker', 'ts']].rename(columns={'ts': 'issue_date'}).copy()
    issue_rows['issue_type'] = issue_type
    issue_rows['description'] = inconsistency_descriptions[issue_type]
    inconsistency_issue_rows.append(issue_rows[['ticker', 'issue_type', 'issue_date', 'description']])

inconsistency_issues = pd.concat(inconsistency_issue_rows, ignore_index=True)
display(inconsistency_issues.groupby('issue_type').size().rename('issue_count').reset_index())
display(inconsistency_issues.head(10))


,issue_type,issue_count
0,adj_close_factor_mismatch,10417
1,open_outside_ohlc_range,5
2,zero_volume,75


,ticker,issue_type,issue_date,description
0,CVX,open_outside_ohlc_range,1962-01-02,Open is outside the low-to-high range.
1,JNJ,open_outside_ohlc_range,1962-01-02,Open is outside the low-to-high range.
2,MCD,open_outside_ohlc_range,1966-07-05,Open is outside the low-to-high range.
3,MRK,open_outside_ohlc_range,1962-01-02,Open is outside the low-to-high range.
4,PG,open_outside_ohlc_range,1962-01-02,Open is outside the low-to-high range.
5,AXP,zero_volume,1972-12-14,Volume is zero; check whether this is a stale-...
6,HON,zero_volume,1982-09-23,Volume is zero; check whether this is a stale-...
7,HON,zero_volume,1982-09-24,Volume is zero; check whether this is a stale-...
8,IBM,zero_volume,1973-10-09,Volume is zero; check whether this is a stale-...
9,IBM,zero_volume,1975-01-27,Volume is zero; check whether this is a stale-...


#### Data Quality Report

Combine the missing-data, outlier, and inconsistency findings in one report. Each issue is a separate row, so one observation can have multiple issues. This report records the diagnosis before repairs; the validation report will be rerun after cleaning.

Severity choices:

- **High**: invalid or impossible prices can make a stock's valuation or price analysis unreliable.
- **Medium**: exact duplicates can distort counts and returns; missing prices or dates and extreme returns need investigation, but do not by themselves prove the observation is wrong.
- **Low**: zero volume can indicate stale data and needs review, but does not alone invalidate the price.


In [13]:
issue_columns = ['ticker', 'issue_type', 'issue_date', 'description', 'severity']

# Combine every detector's rows, keeping each issue as its own report row.
quality_report_parts = [
    missing_data_issues.drop(columns='severity', errors='ignore'),
    duplicate_issue_rows,
    outlier_issues,
    inconsistency_issues,
]
quality_report_before_repair = pd.concat(quality_report_parts, ignore_index=True)

severity_by_issue = {
    'missing_trading_day': 'medium',
    'exact_duplicate': 'medium',
    'extreme_return': 'medium',
    'high_below_close': 'high',
    'low_above_close': 'high',
    'invalid_price': 'high',
    'open_outside_ohlc_range': 'high',
    'close_outside_ohlc_range': 'high',
    'zero_volume': 'low',
    'adj_close_factor_mismatch': 'high',
}
quality_report_before_repair['severity'] = quality_report_before_repair['issue_type'].map(severity_by_issue)

# Missing-value severity depends on which field is missing.
missing_value_mask = quality_report_before_repair['issue_type'].eq('missing_value')
quality_report_before_repair.loc[missing_value_mask, 'severity'] = np.where(
    quality_report_before_repair.loc[missing_value_mask, 'description'].str.contains('volume|adj_close', case=False),
    'medium',
    'high',
)

quality_report_before_repair = quality_report_before_repair[issue_columns].sort_values(
    ['ticker', 'issue_date', 'issue_type'], na_position='last'
).reset_index(drop=True)

print(f'Pre-repair issue rows: {len(quality_report_before_repair):,}')
display(quality_report_before_repair.groupby(['issue_type', 'severity']).size().rename('issue_count').reset_index())
display(quality_report_before_repair.head(10))


Pre-repair issue rows: 83,247


,issue_type,severity,issue_count
0,adj_close_factor_mismatch,high,10417
1,exact_duplicate,medium,29460
2,extreme_return,medium,5056
3,invalid_price,high,18379
4,missing_value,medium,19855
5,open_outside_ohlc_range,high,5
6,zero_volume,low,75


,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26 00:00:00,Daily return is more than 3 sample standard de...,medium
1,AAPL,exact_duplicate,1981-01-02 00:00:00,Exact duplicate record removed before calculat...,medium
2,AAPL,exact_duplicate,1981-01-05 00:00:00,Exact duplicate record removed before calculat...,medium
3,AAPL,invalid_price,1981-01-08 00:00:00,A raw or adjusted equity price is non-finite o...,high
4,AAPL,missing_value,1981-01-08 00:00:00,Missing value in adj_close.,medium
5,AAPL,invalid_price,1981-01-13 00:00:00,A raw or adjusted equity price is non-finite o...,high
6,AAPL,missing_value,1981-01-13 00:00:00,Missing value in adj_close.,medium
7,AAPL,adj_close_factor_mismatch,1981-02-03 00:00:00,Adjusted-close factor differs from the matched...,high
8,AAPL,exact_duplicate,1981-02-10 00:00:00,Exact duplicate record removed before calculat...,medium
9,AAPL,invalid_price,1981-02-12 00:00:00,A raw or adjusted equity price is non-finite o...,high


## 1.3 Clean the Data

### Handle Missing Values

- For missing prices: Use forward-fill (carry last known price)
- For missing volume: Fill with 0 and keep a repair flag (**unknown** volume is not evidence of no trading)
- Drop rows where ALL raw price columns (open, high, low, close) are missing **before** filling
- Missing `adj_close` needs a consistent adjustment convention. This notebook uses matched `(ticker, ts)` rows in the supplied `ohlc` table to obtain a date-specific adjustment factor and reconstructs adjusted close from validated raw close. It does not forward-fill an adjustment factor. Rows that cannot be repaired defensibly are quarantined with a reason.
- For dates with a valid matched reference factor, compare existing `adj_close / close` with that factor. Rebuild mismatched adjusted closes from the validated raw close and matched date-specific factor, and retain mismatch and repair flags. Rows without a matched reference factor are not changed by this check.


In [14]:
def clean_missing_values(frame, reference):
    """Fill supported missing values and return retained, removed, and quarantined rows."""
    raw_price_columns = ['open', 'high', 'low', 'close']
    data = frame.copy()
    data['ticker'] = (data['ticker'].astype('string')
                      .str.replace(r'<[^>]+>', '', regex=True)
                      .str.strip().str.upper())
    data['ts'] = pd.to_datetime(data['ts'])
    data = data.sort_values(['ticker', 'ts']).reset_index(drop=True)

    # Remove exact copies before grouping or filling, and retain their removal record.
    exact_duplicate_mask = data.duplicated(keep='first')
    dropped_exact_duplicates = data.loc[exact_duplicate_mask].copy()
    dropped_exact_duplicates['removal_reason'] = 'Exact duplicate record removed before calculations.'
    data = data.drop_duplicates().copy()
    conflicting_keys = data.duplicated(['ticker', 'ts'], keep=False)
    if conflicting_keys.any():
        raise ValueError('Conflicting (ticker, ts) records need investigation before cleaning.')

    # Remove rows with no raw OHLC information before any forward-fill.
    all_raw_missing = data[raw_price_columns].isna().all(axis=1)
    dropped_all_raw_missing = data.loc[all_raw_missing].copy()
    dropped_all_raw_missing['quarantine_reason'] = 'All raw OHLC prices are missing; dropped before filling.'
    data = data.loc[~all_raw_missing].copy()

    # Forward-fill prices only within each ticker, and retain a flag for every repair.
    for column in raw_price_columns:
        data[f'{column}_repaired'] = data[column].isna()
        data[column] = data.groupby('ticker', sort=False)[column].ffill()

    # A repaired zero marks unknown volume; it does not establish that no trading occurred.
    data['volume_repaired'] = data['volume'].isna()
    data['volume'] = data['volume'].fillna(0)

    # Build a trusted, date-specific factor from the matched reference table.
    reference = reference[['ticker', 'ts', 'close', 'adj_close']].copy()
    reference['ticker'] = (reference['ticker'].astype('string')
                           .str.replace(r'<[^>]+>', '', regex=True)
                           .str.strip().str.upper())
    reference['ts'] = pd.to_datetime(reference['ts'])
    if reference.duplicated(['ticker', 'ts']).any():
        raise ValueError('The ohlc reference has duplicate (ticker, ts) keys; adjustment factors are ambiguous.')
    valid_reference_prices = (
        np.isfinite(reference['close']) & reference['close'].gt(0)
        & np.isfinite(reference['adj_close']) & reference['adj_close'].gt(0)
    )
    reference['adjustment_factor'] = np.nan
    reference.loc[valid_reference_prices, 'adjustment_factor'] = (
        reference.loc[valid_reference_prices, 'adj_close']
        / reference.loc[valid_reference_prices, 'close']
    )
    data = data.merge(
        reference[['ticker', 'ts', 'adjustment_factor']],
        on=['ticker', 'ts'],
        how='left',
        validate='many_to_one',
    )

    # Compare existing adjustment factors when a valid date-specific reference is available.
    raw_close_valid = (
        np.isfinite(data['close']) & data['close'].gt(0)
        & data['close'].between(data['low'], data['high'])
    )
    current_factor_valid = (
        np.isfinite(data['adj_close']) & data['adj_close'].gt(0)
        & raw_close_valid
        & np.isfinite(data['adjustment_factor']) & data['adjustment_factor'].gt(0)
    )
    current_adjustment_factor = data['adj_close'] / data['close']
    factor_matches_reference = np.isclose(
        current_adjustment_factor, data['adjustment_factor'], rtol=1e-6, atol=1e-8
    )
    data['adj_close_factor_mismatch'] = current_factor_valid & ~factor_matches_reference

    # Rebuild missing or mismatched adjusted closes, and refresh after a raw-close repair.
    needs_adjusted_repair = (
        data['adj_close'].isna() | data['close_repaired'] | data['adj_close_factor_mismatch']
    )
    can_rebuild_adjusted = (
        needs_adjusted_repair & raw_close_valid
        & np.isfinite(data['adjustment_factor']) & data['adjustment_factor'].gt(0)
    )
    data['adj_close_repaired'] = can_rebuild_adjusted
    data.loc[can_rebuild_adjusted, 'adj_close'] = (
        data.loc[can_rebuild_adjusted, 'close']
        * data.loc[can_rebuild_adjusted, 'adjustment_factor']
    )

    # Quarantine unresolved leading gaps and repairs that leave invalid OHLC bounds.
    raw_still_missing = data[raw_price_columns].isna().any(axis=1)
    adjusted_still_missing = needs_adjusted_repair & data['adj_close'].isna()
    bounds_valid = (
        data['low'].le(data['open']) & data['open'].le(data['high'])
        & data['low'].le(data['close']) & data['close'].le(data['high'])
    )
    filled_raw_prices = data[[f'{column}_repaired' for column in raw_price_columns]].any(axis=1)
    repaired_bounds_invalid = filled_raw_prices & ~bounds_valid
    quarantine_mask = raw_still_missing | adjusted_still_missing | repaired_bounds_invalid

    data['quarantine_reason'] = ''
    data.loc[raw_still_missing, 'quarantine_reason'] += 'Unresolved leading raw-price gap; '
    data.loc[adjusted_still_missing, 'quarantine_reason'] += 'No validated raw close and matched adjustment factor; '
    data.loc[repaired_bounds_invalid, 'quarantine_reason'] += 'Forward-filled raw prices violate OHLC bounds; '
    quarantined_rows = data.loc[quarantine_mask].copy()
    cleaned_rows = data.loc[~quarantine_mask].drop(columns='adjustment_factor').copy()
    return cleaned_rows, dropped_exact_duplicates, dropped_all_raw_missing, quarantined_rows


clean_missing, dropped_exact_duplicates, dropped_all_raw_missing, quarantined_missing = clean_missing_values(
    ohlc_hw, reference_prices
)

print(f'Rows retained after missing-value handling: {len(clean_missing):,}')
print(f'Exact duplicate rows removed before filling: {len(dropped_exact_duplicates):,}')
print(f'Rows dropped because all raw OHLC prices were missing: {len(dropped_all_raw_missing):,}')
print(f'Rows quarantined as unresolved: {len(quarantined_missing):,}')
print('Repair counts:')
display(clean_missing[['open_repaired', 'high_repaired', 'low_repaired', 'close_repaired',
                       'adj_close_factor_mismatch', 'adj_close_repaired', 'volume_repaired']].sum().rename('repaired_rows'))


Rows retained after missing-value handling: 368,251
Exact duplicate rows removed before filling: 29,460
Rows dropped because all raw OHLC prices were missing: 0
Rows quarantined as unresolved: 0
Repair counts:


open_repaired                    0
high_repaired                    0
low_repaired                     0
close_repaired                   0
adj_close_factor_mismatch    10417
adj_close_repaired           28791
volume_repaired                  0
Name: repaired_rows, dtype: int64

### Handle Outliers

**Impossible OHLC relationships:** Replace the offending raw price with `NaN`, then forward-fill within the same ticker (Option 1). If there is no previous value or the repaired row still violates OHLC bounds, quarantine it instead of inventing a price. Rebuild adjusted close from the matched date-specific factor whenever close is repaired.

**Extreme returns:** Calculate the review flags after exact duplicate records are removed. The supplied database has no documented event or vendor notes that identify the cause of every flagged return. For example, TRV's raw close changes from 4.61 to 9.22 and back to 4.69 over three supplied observations, which suggests a one-observation quote anomaly; the cause is not confirmed by this database. Retain all statistical flags for review and do not interpolate or label them as confirmed split, merger, news, or error events. The 3-standard-deviation screen alone is not a deletion rule.


In [15]:
def clean_outliers(frame, reference):
    """Repair impossible OHLC fields by ticker-wise forward-fill; quarantine unresolved rows."""
    data = frame.copy().sort_values(['ticker', 'ts']).reset_index(drop=True)
    raw_price_columns = ['open', 'high', 'low', 'close']

    # Preserve the retrospective return screen before changing any price values.
    valid_close = data['close'].where(np.isfinite(data['close']) & data['close'].gt(0))
    original_return = valid_close.groupby(data['ticker']).pct_change(fill_method=None)
    return_groups = original_return.groupby(data['ticker'])
    mean_return = return_groups.transform('mean')
    std_return = return_groups.transform('std', ddof=1)
    data['daily_return'] = original_return
    data['extreme_return_flag'] = (
        original_return.notna() & std_return.notna() & std_return.gt(0)
        & (original_return - mean_return).abs().gt(3 * std_return)
    )

    # Identify the specific OHLC fields that violate positivity or today's price range.
    bad_price_masks = {
        'open': (~np.isfinite(data['open']) | data['open'].le(0)
                 | data['open'].lt(data['low']) | data['open'].gt(data['high'])),
        'high': (~np.isfinite(data['high']) | data['high'].le(0)
                 | data['high'].lt(data['open']) | data['high'].lt(data['close'])),
        'low': (~np.isfinite(data['low']) | data['low'].le(0)
                | data['low'].gt(data['open']) | data['low'].gt(data['close'])),
        'close': (~np.isfinite(data['close']) | data['close'].le(0)
                  | data['close'].lt(data['low']) | data['close'].gt(data['high'])),
    }
    for column, bad_mask in bad_price_masks.items():
        flag_column = f'{column}_repaired'
        prior_flag = data[flag_column] if flag_column in data else False
        data[flag_column] = prior_flag | bad_mask
        data.loc[bad_mask, column] = np.nan
        data[column] = data.groupby('ticker', sort=False)[column].ffill()

    # Keep adjusted prices on the same convention after any close repair.
    reference = reference[['ticker', 'ts', 'close', 'adj_close']].copy()
    reference['ticker'] = (reference['ticker'].astype('string')
                           .str.replace(r'<[^>]+>', '', regex=True)
                           .str.strip().str.upper())
    reference['ts'] = pd.to_datetime(reference['ts'])
    if reference.duplicated(['ticker', 'ts']).any():
        raise ValueError('The ohlc reference has duplicate (ticker, ts) keys; adjustment factors are ambiguous.')
    factor_is_valid = (
        np.isfinite(reference['close']) & reference['close'].gt(0)
        & np.isfinite(reference['adj_close']) & reference['adj_close'].gt(0)
    )
    reference['adjustment_factor'] = np.nan
    reference.loc[factor_is_valid, 'adjustment_factor'] = (
        reference.loc[factor_is_valid, 'adj_close'] / reference.loc[factor_is_valid, 'close']
    )
    data = data.merge(
        reference[['ticker', 'ts', 'adjustment_factor']],
        on=['ticker', 'ts'], how='left', validate='many_to_one',
    )
    close_repaired = data['close_repaired']
    close_valid = (
        np.isfinite(data['close']) & data['close'].gt(0)
        & data['close'].between(data['low'], data['high'])
    )
    can_rebuild_adj = (
        close_repaired & close_valid
        & np.isfinite(data['adjustment_factor']) & data['adjustment_factor'].gt(0)
    )
    if 'adj_close_repaired' not in data:
        data['adj_close_repaired'] = False
    data['adj_close_repaired'] = data['adj_close_repaired'] | can_rebuild_adj
    data.loc[can_rebuild_adj, 'adj_close'] = (
        data.loc[can_rebuild_adj, 'close'] * data.loc[can_rebuild_adj, 'adjustment_factor']
    )

    bounds_valid = (
        data['low'].le(data['open']) & data['open'].le(data['high'])
        & data['low'].le(data['close']) & data['close'].le(data['high'])
    )
    raw_unresolved = data[raw_price_columns].isna().any(axis=1)
    raw_nonpositive = data[raw_price_columns].le(0).any(axis=1)
    bad_adjusted_close = ~np.isfinite(data['adj_close']) | data['adj_close'].le(0)
    close_repair_unresolved = close_repaired & ~can_rebuild_adj
    quarantine_mask = raw_unresolved | raw_nonpositive | ~bounds_valid | bad_adjusted_close | close_repair_unresolved

    data['quarantine_reason'] = ''
    data.loc[raw_unresolved | raw_nonpositive, 'quarantine_reason'] += 'Unresolved or nonpositive raw price after forward-fill; '
    data.loc[~bounds_valid, 'quarantine_reason'] += 'OHLC bounds remain invalid after repair; '
    data.loc[bad_adjusted_close | close_repair_unresolved, 'quarantine_reason'] += 'Adjusted close is invalid or could not be rebuilt from a matched factor; '
    quarantined_rows = data.loc[quarantine_mask].copy()
    cleaned_rows = data.loc[~quarantine_mask].drop(columns='adjustment_factor').copy()
    # Recalculate returns after price repairs; keep extreme_return_flag as the pre-repair diagnosis.
    cleaned_rows['daily_return'] = cleaned_rows.groupby('ticker')['close'].pct_change(fill_method=None)
    return cleaned_rows, quarantined_rows


clean_after_outliers, quarantined_outliers = clean_outliers(clean_missing, reference_prices)
print(f'Rows retained after outlier handling: {len(clean_after_outliers):,}')
print(f'Rows quarantined after outlier handling: {len(quarantined_outliers):,}')
print('Outlier repair flags:')
display(clean_after_outliers[['open_repaired', 'high_repaired', 'low_repaired', 'close_repaired',
                              'adj_close_repaired', 'extreme_return_flag']].sum().rename('flagged_rows'))
if not quarantined_outliers.empty:
    display(quarantined_outliers[['ticker', 'ts', 'open_repaired', 'quarantine_reason']].head(10))


Rows retained after outlier handling: 368,246
Rows quarantined after outlier handling: 5
Outlier repair flags:


open_repaired              0
high_repaired              0
low_repaired               0
close_repaired             0
adj_close_repaired     28791
extreme_return_flag     5056
Name: flagged_rows, dtype: int64

,ticker,ts,open_repaired,quarantine_reason
87103,CVX,1962-01-02,True,Unresolved or nonpositive raw price after forw...
167733,JNJ,1962-01-02,True,Unresolved or nonpositive raw price after forw...
213412,MCD,1966-07-05,True,Unresolved or nonpositive raw price after forw...
243893,MRK,1962-01-02,True,Unresolved or nonpositive raw price after forw...
287040,PG,1962-01-02,True,Unresolved or nonpositive raw price after forw...


### Validation

Validate the retained rows after repairs, account for removed and quarantined rows, and compare raw-data statistics before and after cleaning. Only rows that pass validation are saved to `ohlc_hw_cleaned`; the original `ohlc_hw` table remains unchanged.


In [16]:
raw_price_columns = ['open', 'high', 'low', 'close']
all_price_columns = raw_price_columns + ['adj_close']
cleaned_for_validation = clean_after_outliers.copy()

# Validate all retained raw and adjusted prices.
retained_prices = cleaned_for_validation[all_price_columns]
assert np.isfinite(retained_prices.to_numpy(dtype=float)).all(), 'Retained prices contain missing or non-finite values.'
assert retained_prices.gt(0).all().all(), 'Retained prices contain nonpositive values.'

# Validate unique keys, volume, dates, and OHLC relationships.
assert not cleaned_for_validation.duplicated(['ticker', 'ts']).any(), 'Duplicate (ticker, ts) keys remain.'
assert np.isfinite(cleaned_for_validation['volume'].to_numpy(dtype=float)).all(), 'Volume contains non-finite values.'
assert cleaned_for_validation['volume'].ge(0).all(), 'Volume contains negative values.'
assert cleaned_for_validation['ts'].notna().all(), 'A retained row has no timestamp.'
assert (
    cleaned_for_validation['low'].le(cleaned_for_validation['open'])
    & cleaned_for_validation['open'].le(cleaned_for_validation['high'])
    & cleaned_for_validation['low'].le(cleaned_for_validation['close'])
    & cleaned_for_validation['close'].le(cleaned_for_validation['high'])
).all(), 'A retained row violates OHLC bounds.'

# Verify that every adjusted close rebuilt during repair uses its matched date-specific factor.
reference_factors = reference_prices[['ticker', 'ts', 'close', 'adj_close']].copy()
reference_factors['ticker'] = (reference_factors['ticker'].astype('string')
                                .str.replace(r'<[^>]+>', '', regex=True)
                                .str.strip().str.upper())
reference_factors['adjustment_factor'] = reference_factors['adj_close'] / reference_factors['close']
repaired_adjusted = cleaned_for_validation.loc[cleaned_for_validation['adj_close_repaired']].merge(
    reference_factors[['ticker', 'ts', 'adjustment_factor']],
    on=['ticker', 'ts'], how='left', validate='one_to_one',
)
assert repaired_adjusted['adjustment_factor'].notna().all(), 'A repaired adjusted close has no matched reference factor.'
assert np.allclose(
    repaired_adjusted['adj_close'],
    repaired_adjusted['close'] * repaired_adjusted['adjustment_factor'],
    rtol=1e-10, atol=1e-12,
), 'A repaired adjusted close is inconsistent with its matched factor.'

# Validate the same adjustment convention for all retained rows with a valid matched factor.
valid_reference_factors = reference_factors.loc[
    np.isfinite(reference_factors['adjustment_factor'])
    & reference_factors['adjustment_factor'].gt(0),
    ['ticker', 'ts', 'adjustment_factor'],
]
matched_adjusted = cleaned_for_validation.merge(
    valid_reference_factors, on=['ticker', 'ts'], how='inner', validate='one_to_one'
)
assert np.allclose(
    matched_adjusted['adj_close'],
    matched_adjusted['close'] * matched_adjusted['adjustment_factor'],
    rtol=1e-6, atol=1e-8,
), 'A retained adjusted close disagrees with its matched date-specific factor.'
print(f'Adjusted closes repaired from matched factors: {int(cleaned_for_validation["adj_close_repaired"].sum()):,}')
print(f'Retained rows checked against matched factors: {len(matched_adjusted):,}')

removed_row_count = len(dropped_exact_duplicates) + len(dropped_all_raw_missing)
quarantined_row_count = len(quarantined_missing) + len(quarantined_outliers)
row_accounting = pd.DataFrame({
    'status': ['retained', 'removed', 'quarantined', 'input_total'],
    'rows': [
        len(cleaned_for_validation),
        removed_row_count,
        quarantined_row_count,
        len(ohlc_hw),
    ],
})
assert len(cleaned_for_validation) + removed_row_count + quarantined_row_count == len(ohlc_hw), (
    'Retained, removed, and quarantined counts do not account for every input row.'
)

print('All retained-row validation checks passed.')
display(row_accounting)

# Compare before/after price statistics and missing counts.
before_stats = ohlc_hw[all_price_columns].describe().T
before_stats['missing_values'] = ohlc_hw[all_price_columns].isna().sum()
before_stats['dataset'] = 'before'
after_stats = cleaned_for_validation[all_price_columns].describe().T
after_stats['missing_values'] = cleaned_for_validation[all_price_columns].isna().sum()
after_stats['dataset'] = 'after'
before_after_stats = pd.concat([before_stats, after_stats]).rename_axis('price_column').reset_index()
display(before_after_stats[['dataset', 'price_column', 'count', 'missing_values', 'mean', 'std', 'min', 'max']])


Adjusted closes repaired from matched factors: 28,791
Retained rows checked against matched factors: 368,246
All retained-row validation checks passed.


,status,rows
0,retained,368246
1,removed,29460
2,quarantined,5
3,input_total,397711


,dataset,price_column,count,missing_values,mean,std,min,max
0,before,open,397711.0,0,46.285479,67.132594,0.000000,604.260000
1,before,high,397711.0,0,46.747660,67.778210,0.005208,608.630000
2,before,low,397711.0,0,45.816279,66.477178,0.004801,597.880000
3,before,close,397711.0,0,46.292903,67.141879,0.005208,605.400000
4,before,adj_close,377856.0,19855,51.652250,194.584045,0.002945,10442.510783
5,after,open,368246.0,0,46.309394,67.143254,0.005208,604.260000
6,after,high,368246.0,0,46.771824,67.789319,0.005208,608.630000
7,after,low,368246.0,0,45.840375,66.489450,0.004801,597.880000
8,after,close,368246.0,0,46.317325,67.154939,0.005208,605.400000
9,after,adj_close,368246.0,0,37.274670,63.587872,0.002945,605.400000


In [17]:
# Store timestamps as ISO date strings and write only the new cleaned table.
cleaned_table = cleaned_for_validation.copy()
cleaned_table['ts'] = pd.to_datetime(cleaned_table['ts']).dt.strftime('%Y-%m-%d')

with closing(sqlite3.connect(database)) as connection:
    raw_count_before_write = pd.read_sql_query('SELECT COUNT(*) AS n FROM ohlc_hw', connection).iloc[0, 0]
    cleaned_table.to_sql('ohlc_hw_cleaned', connection, if_exists='replace', index=False)
    raw_count_after_write = pd.read_sql_query('SELECT COUNT(*) AS n FROM ohlc_hw', connection).iloc[0, 0]
    saved_table = pd.read_sql_query('SELECT * FROM ohlc_hw_cleaned', connection)

assert raw_count_before_write == raw_count_after_write == len(ohlc_hw), 'The raw table row count changed.'
assert len(saved_table) == len(cleaned_table), 'The saved cleaned table has the wrong number of rows.'
assert saved_table['ts'].str.fullmatch(r'\d{4}-\d{2}-\d{2}').all(), 'Saved dates are not ISO YYYY-MM-DD strings.'
assert not saved_table.duplicated(['ticker', 'ts']).any(), 'The saved cleaned table has duplicate keys.'
print(f'Saved ohlc_hw_cleaned: {len(saved_table):,} rows.')
print(f'Raw ohlc_hw remains at {raw_count_after_write:,} rows.')


Saved ohlc_hw_cleaned: 368,246 rows.
Raw ohlc_hw remains at 397,711 rows.
